In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import optuna

In [2]:
data_path = '../src/clean_data.csv'
random_state=42

In [3]:
df = pd.read_csv(data_path, index_col=0)
df.head()

,record_id,r_peak,annotation_sample,type,dist_from_annotation,x_0,x_1,x_2,x_3,x_4,...,x_171,x_172,x_173,x_174,x_175,x_176,x_177,x_178,x_179,anomaly
0,100,370,370.0,N,0.0,-0.280,-0.295,-0.305,-0.315,-0.305,...,-0.455,-0.450,-0.435,-0.445,-0.450,-0.455,-0.470,-0.460,-0.455,0
1,100,663,662.0,N,1.0,-0.335,-0.330,-0.335,-0.345,-0.345,...,-0.435,-0.430,-0.435,-0.420,-0.420,-0.420,-0.420,-0.425,-0.425,0
2,100,947,946.0,N,1.0,-0.345,-0.345,-0.345,-0.340,-0.310,...,-0.375,-0.365,-0.370,-0.385,-0.395,-0.385,-0.385,-0.370,-0.365,0
3,100,1231,1231.0,N,0.0,-0.330,-0.320,-0.290,-0.290,-0.300,...,-0.470,-0.455,-0.460,-0.440,-0.435,-0.425,-0.430,-0.445,-0.450,0
4,100,1515,1515.0,N,0.0,-0.320,-0.310,-0.305,-0.305,-0.275,...,-0.445,-0.440,-0.435,-0.430,-0.430,-0.450,-0.445,-0.445,-0.435,0


In [4]:
feature_columns = [f'x_{i}' for i in range(180)]

In [5]:
print(len(feature_columns))

180


In [6]:
record_ids = sorted(df['record_id'].unique())

patient_groups = {
    record_id: 201 if record_id == 202 else record_id
    for record_id in record_ids
}

unique_patients = sorted(set(patient_groups.values()))

In [7]:
train_patients, temp_patients = train_test_split(
    unique_patients,
    test_size=0.4,
    random_state=random_state
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.5,
    random_state=random_state
)

print("Train:", len(train_patients))
print("Validation:", len(val_patients))
print("Test:", len(test_patients))

Train: 27
Validation: 9
Test: 9


In [8]:
train_ids = [
    rid for rid in record_ids
    if patient_groups[rid] in train_patients
]

val_ids = [
    rid for rid in record_ids
    if patient_groups[rid] in val_patients
]

test_ids = [
    rid for rid in record_ids
    if patient_groups[rid] in test_patients
]

In [9]:
train_df_all = df[
    df['record_id'].isin(train_ids)
].copy()

train_df = train_df_all[
    train_df_all['anomaly'] == 0
].copy()

val_df = df[
    df['record_id'].isin(val_ids)
].copy()

test_df = df[
    df['record_id'].isin(test_ids)
].copy()

In [10]:
for name, dataset in [
    ('Train', train_df_all),
    ('Validation', val_df),
    ('Test', test_df),
]:
    print(
        name,
        'records:', dataset['record_id'].nunique(),
        'beats:', len(dataset),
        'anomaly %:', round(dataset['anomaly'].mean() * 100, 2)
    )

Train records: 28 beats: 57551 anomaly %: 18.35
Validation records: 9 beats: 19544 anomaly %: 19.4
Test records: 9 beats: 16411 anomaly %: 35.63


In [11]:
pd.DataFrame({
    'train': train_df_all['type'].value_counts(),
    'validation': val_df['type'].value_counts(),
    'test': test_df['type'].value_counts()
})

,train,validation,test
type,,,
A,649,174.0,1392
L,4492,NaN,2029
N,46992,15753.0,10563
R,3356,2165.0,1736
V,2062,1452.0,691


In [12]:
df[df['type']=='L'].groupby('record_id').size()

record_id
109    2491
111    2016
207      13
214    2001
dtype: int64

In [13]:
df[df['type']=='A'].groupby('record_id').size()

record_id
100      33
101       3
103       2
108       3
112       2
114      10
116       1
117       1
118      95
121       1
124       2
200      30
201      24
202      30
205       3
209     232
213      21
215       2
219       7
220      73
222     181
223      71
228       3
231       1
232    1379
233       5
dtype: int64

In [14]:
df[df['type']=='R'].groupby('record_id').size()

record_id
118    2165
124    1531
207      85
212    1825
231    1254
232     397
dtype: int64

In [15]:
df[df['type']=='V'].groupby('record_id').size()

record_id
105     41
106    419
107     16
108      9
109     29
111      1
114      2
116    106
118     16
119    444
123      3
124      8
200    116
201     15
202      1
203     43
205     16
207     40
208    826
209      1
210     38
213    213
214    128
215    141
217    155
219     64
221    335
223    451
228    288
231      1
233    236
234      3
dtype: int64

In [26]:
fixed_train_ids = [109, 118, 212, 232, 208, 106]
fixed_val_ids = [111, 124, 209, 119]
fixed_test_ids = [214, 231, 222, 223]

fixed_ids = set(fixed_train_ids + fixed_val_ids + fixed_test_ids)

In [27]:
remaining_ids = sorted(set(record_ids) - fixed_ids)
print('Pozostałe rekordy: ', len(remaining_ids))

Pozostałe rekordy:  32


In [28]:
remaining_groups = sorted({
    patient_groups[rid] for rid in remaining_ids
})